In [28]:
import json
from pathlib import Path
from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
DATA_PATH = Path("../data/qwen_train_data.jsonl")

In [29]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [30]:
def load_jsonl(path):
    examples = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            examples.append(json.loads(line))
    return examples

examples = load_jsonl(DATA_PATH)

print("Number of examples:", len(examples))
print(examples[0].keys())

Number of examples: 1030
dict_keys(['text'])


In [31]:
ex1 = examples[0]["text"]
ex2 = examples[1]["text"]

print(len(ex1), len(ex2))

1828 3920


In [32]:
enc1 = tokenizer(ex1)
enc2 = tokenizer(ex2)

print(len(enc1["input_ids"]))
print(len(enc2["input_ids"]))

411
822


In [33]:
batch = tokenizer(
    [ex1, ex2],
    padding=True,
    return_tensors="pt"
)

In [34]:
batch["input_ids"].shape

torch.Size([2, 822])

In [35]:
print(batch["attention_mask"])

tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 1, 1, 1]])


In [36]:
print(batch["input_ids"])

tensor([[151644,   8948,    198,  ..., 151643, 151643, 151643],
        [151644,   8948,    198,  ...,     13, 151645,    198]])


In [37]:
tokenizer.pad_token
tokenizer.pad_token_id

151643

In [38]:
pad_id = tokenizer.pad_token_id
print(tokenizer.decode([pad_id]))

<|endoftext|>


In [39]:
for token_id, mask in zip(
    batch["input_ids"][0][-30:],
    batch["attention_mask"][0][-30:]
):
    print(token_id.item(), mask.item(), repr(tokenizer.decode([token_id])))

151643 0 '<|endoftext|>'
151643 0 '<|endoftext|>'
151643 0 '<|endoftext|>'
151643 0 '<|endoftext|>'
151643 0 '<|endoftext|>'
151643 0 '<|endoftext|>'
151643 0 '<|endoftext|>'
151643 0 '<|endoftext|>'
151643 0 '<|endoftext|>'
151643 0 '<|endoftext|>'
151643 0 '<|endoftext|>'
151643 0 '<|endoftext|>'
151643 0 '<|endoftext|>'
151643 0 '<|endoftext|>'
151643 0 '<|endoftext|>'
151643 0 '<|endoftext|>'
151643 0 '<|endoftext|>'
151643 0 '<|endoftext|>'
151643 0 '<|endoftext|>'
151643 0 '<|endoftext|>'
151643 0 '<|endoftext|>'
151643 0 '<|endoftext|>'
151643 0 '<|endoftext|>'
151643 0 '<|endoftext|>'
151643 0 '<|endoftext|>'
151643 0 '<|endoftext|>'
151643 0 '<|endoftext|>'
151643 0 '<|endoftext|>'
151643 0 '<|endoftext|>'
151643 0 '<|endoftext|>'


In [40]:
labels = batch["input_ids"].clone()
labels[batch["attention_mask"] == 0] = -100

print(labels[0][-30:])
print(labels[1][-30:])

tensor([-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100])
tensor([   264,  14977,  19415,    369,    432,     13,   4695,    264,   1661,
          2310,  10008,     12,  92350,  17654,   1075,    856,     23,     21,
            62,     21,     19,     11,    429,    594,   2494,    582,   3535,
            13, 151645,    198])


In [41]:
for i in range(2):
    seq_len = int(batch["attention_mask"][i].sum())
    ids = batch["input_ids"][i]
    print(f"\nExample {i}:")
    for j in range(seq_len - 10, seq_len):
        token_id = ids[j].item()
        print(j, token_id, repr(tokenizer.decode([token_id])))


Example 0:
401 303 'nd'
402 1578 ' ed'
403 11 ','
404 4182 ' Ne'
405 324 'ur'
406 24202 'onal'
407 21248 ' Migration'
408 568 ').'
409 151645 '<|im_end|>'
410 198 '\n'

Example 1:
812 19 '4'
813 11 ','
814 429 ' that'
815 594 "'s"
816 2494 ' something'
817 582 ' we'
818 3535 ' understand'
819 13 '.'
820 151645 '<|im_end|>'
821 198 '\n'
